In [1]:
import pandas as pd
import numpy as np
 
pd.set_option("display.max_columns", None)

In [3]:
department_raw = pd.read_csv("C:\\Users\\mrity\\Downloads\\Student Device Usage and Engagement Analysis\\CSV Dataset\\Department.csv")
devices_info_raw = pd.read_csv("C:\\Users\\mrity\\Downloads\\Student Device Usage and Engagement Analysis\\CSV Dataset\\Devices Info.csv")
devices_raw = pd.read_csv("C:\\Users\\mrity\\Downloads\\Student Device Usage and Engagement Analysis\\CSV Dataset\\Devices.csv")
student_engagement_raw = pd.read_csv("C:\\Users\\mrity\\Downloads\\Student Device Usage and Engagement Analysis\\CSV Dataset\\Student Engagement.csv")


In [4]:
def clean_text(series: pd.Series) -> pd.Series:
    """Trim whitespace and title-case a text column."""
    return series.astype(str).str.strip().str.title()
 
 
def report_dupes(df: pd.DataFrame, subset, name: str):
    dupes = df[df.duplicated(subset=subset, keep=False)]
    if not dupes.empty:
        print(f"[{name}] {len(dupes)} duplicate rows found on {subset}:")
        print(dupes)
    return dupes

In [5]:
department = department_raw.copy()
 
# Trim + standardize casing
department["Department"] = clean_text(department["Department"])
department["Field"] = clean_text(department["Field"])
 
# Drop rows with missing key
department = department.dropna(subset=["Department"])
 
# Flag label mismatches (e.g. 'Sciences' vs 'Science') for manual review
mismatches = department[department["Department"].str.lower() != department["Field"].str.lower()]
if not mismatches.empty:
    print("[Department] Department/Field label mismatches to review:")
    print(mismatches)
 
# Dedupe on Department (primary key)
report_dupes(department, ["Department"], "Department")
department = department.drop_duplicates(subset=["Department"], keep="first").reset_index(drop=True)

[Department] Department/Field label mismatches to review:
  Department    Field
4   Sciences  Science


In [6]:
devices_info = devices_info_raw.copy()
 
devices_info["DeviceType"] = clean_text(devices_info["DeviceType"])
devices_info["Category"] = clean_text(devices_info["Category"])
devices_info["Connectivity"] = devices_info["Connectivity"].astype(str).str.strip()
 
# Coerce numerics, turning bad values into NaN instead of crashing
devices_info["Avg_Battery_Life_Hours"] = pd.to_numeric(devices_info["Avg_Battery_Life_Hours"], errors="coerce")
devices_info["Screen_Size_Inches"] = pd.to_numeric(devices_info["Screen_Size_Inches"], errors="coerce")
 
# Remove impossible values (battery life / screen size can't be <= 0)
bad_values = devices_info[
    (devices_info["Avg_Battery_Life_Hours"] <= 0) | (devices_info["Screen_Size_Inches"] <= 0)
]
if not bad_values.empty:
    print("[DevicesInfo] Rows dropped for non-positive battery life / screen size:")
    print(bad_values)
devices_info = devices_info[
    (devices_info["Avg_Battery_Life_Hours"] > 0) & (devices_info["Screen_Size_Inches"] > 0)
]

In [7]:
report_dupes(devices_info, ["DeviceType"], "DevicesInfo")
devices_info = devices_info.drop_duplicates(subset=["DeviceType"], keep="first").reset_index(drop=True)

In [8]:
devices = devices_raw.copy()
 
devices["DeviceType"] = clean_text(devices["DeviceType"])
devices["Category"] = clean_text(devices["Category"])
devices = devices.dropna(subset=["DeviceType"])
devices = devices.drop_duplicates(subset=["DeviceType", "Category"], keep="first").reset_index(drop=True)
 
# Sanity check: every DeviceType here should exist in Devices Info
orphans = devices[~devices["DeviceType"].isin(devices_info["DeviceType"])]
if not orphans.empty:
    print("[Devices] DeviceTypes not found in DevicesInfo:")
    print(orphans)

In [9]:
department_raw.head()

,Department,Field
0,Arts,Arts
1,Business,Business
2,Medical,Medical
3,Engineering,Engineering
4,Sciences,Science


In [10]:
student_engagement = student_engagement_raw.copy()
 
# Trim + standardize text fields
student_engagement["DeviceType"] = clean_text(student_engagement["DeviceType"])
student_engagement["Department"] = clean_text(student_engagement["Department"])
student_engagement["AgeGroup"] = student_engagement["AgeGroup"].astype(str).str.strip()
student_engagement["Region"] = clean_text(student_engagement["Region"])
 
# Coerce numeric columns
numeric_cols = [
    "StudentID", "Study_Hours", "Engagement_Score",
    "Performance_Rating", "Internet_Performance_Impact",
]
for col in numeric_cols:
    student_engagement[col] = pd.to_numeric(student_engagement[col], errors="coerce")
 
# Drop rows missing required keys
student_engagement = student_engagement.dropna(subset=["StudentID", "DeviceType"])
student_engagement["StudentID"] = student_engagement["StudentID"].astype(int)
 
# Dedupe on StudentID (keep first occurrence)
report_dupes(student_engagement, ["StudentID"], "StudentEngagement")
student_engagement = student_engagement.drop_duplicates(subset=["StudentID"], keep="first").reset_index(drop=True)
 
# Range checks: scores/ratings should be 0-100
out_of_range = student_engagement[
    ~student_engagement["Engagement_Score"].between(0, 100)
    | ~student_engagement["Performance_Rating"].between(0, 100)
]
if not out_of_range.empty:
    print("[StudentEngagement] Rows dropped for out-of-range scores/ratings:")
    print(out_of_range)
student_engagement = student_engagement[
    student_engagement["Engagement_Score"].between(0, 100)
    & student_engagement["Performance_Rating"].between(0, 100)
]

In [11]:
neg_hours = student_engagement["Study_Hours"] < 0
if neg_hours.any():
    print(f"[StudentEngagement] {neg_hours.sum()} negative Study_Hours values set to NaN")
    student_engagement.loc[neg_hours, "Study_Hours"] = np.nan
 
# Referential integrity checks
bad_dept = student_engagement[~student_engagement["Department"].isin(department["Department"])]
if not bad_dept.empty:
    print("[StudentEngagement] Rows with Department not found in Department table:")
    print(bad_dept[["StudentID", "Department"]])
 
bad_device = student_engagement[~student_engagement["DeviceType"].isin(devices_info["DeviceType"])]
if not bad_device.empty:
    print("[StudentEngagement] Rows with DeviceType not found in DevicesInfo table:")
    print(bad_device[["StudentID", "DeviceType"]])

In [12]:
analysis_df = (
    student_engagement
    .merge(department, on="Department", how="inner")
    .merge(devices_info, on="DeviceType", how="inner", suffixes=("", "_device"))
)
 
# Reorder / select final columns
analysis_df = analysis_df[
    [
        "StudentID", "Department", "Field", "AgeGroup", "Region",
        "DeviceType", "Category", "Avg_Battery_Life_Hours",
        "Connectivity", "Screen_Size_Inches",
        "Study_Hours", "Engagement_Score", "Performance_Rating",
        "Internet_Performance_Impact",
    ]
].rename(columns={"Category": "Device_Category"})
 
print(f"\nFinal analysis table: {len(analysis_df)} rows, "
      f"{analysis_df['StudentID'].nunique()} distinct students")
print(analysis_df.head())


Final analysis table: 3482 rows, 3482 distinct students
   StudentID   Department        Field  AgeGroup Region     DeviceType  \
0          1         Arts         Arts     18-22  North        Desktop   
1          2         Arts         Arts  Under 18  South  Hybrid Device   
2          3  Engineering  Engineering     18-22  North         Tablet   
3          4     Business     Business     23-30   West  Hybrid Device   
4          5         Arts         Arts  Under 18   West  Hybrid Device   

  Device_Category  Avg_Battery_Life_Hours Connectivity  Screen_Size_Inches  \
0       Computing                       6     Ethernet                17.3   
1          Hybrid                       9    WiFi & 5G                13.5   
2          Tablet                      12         WiFi                10.1   
3          Hybrid                       9    WiFi & 5G                13.5   
4          Hybrid                       9    WiFi & 5G                13.5   

   Study_Hours  Engagement_Sc

In [13]:
analysis_df.to_csv("vw_StudentDeviceAnalysis.csv", index=False)
department.to_csv("dim_Department_clean.csv", index=False)
devices_info.to_csv("dim_DevicesInfo_clean.csv", index=False)
student_engagement.to_csv("fact_StudentEngagement_clean.csv", index=False)